In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import docling
from pathlib import Path

from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from langchain_docling.loader import DoclingLoader, ExportType

from langchain_text_splitters.markdown import MarkdownHeaderTextSplitter
from langchain_openai.embeddings import OpenAIEmbeddings
import numpy as np

In [ ]:
# Get the file path to the PDF file
FILE_PATH = Path.cwd() / "RegulationsfortheFIFAWorldCup26_noimages.pdf"

# Set your pipeline options for docling
pipeline_options = PdfPipelineOptions(
    allow_external_plugins=True,
    do_ocr=False,
    do_table_structure=True,
    force_backend_text=True,
    generate_parsed_pages=False,
)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options,
            backend=PyPdfiumDocumentBackend,
        )
    }
)

loader = DoclingLoader(
    file_path=str(FILE_PATH),
    converter=converter,
    export_type=ExportType.MARKDOWN,
)

docs = loader.load()
markdown = docs[0].page_content

print("documents loaded:", len(docs))
print("markdown length:", len(markdown))

In [ ]:
md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on = [("#", "Header 1"), 
                                                                ("##", "Header 2")])
pages_md_split = md_splitter.split_text(docs[0].page_content)                                                                

In [ ]:
print(pages_md_split[0].page_content)
print('********')
print(pages_md_split[1].page_content)
print('********')
print(pages_md_split[2].page_content)


In [ ]:
embedding = OpenAIEmbeddings(model = "text-embedding-3-small")

In [ ]:
vector0 = embedding.embed_query(pages_md_split[0].page_content)
vector1 = embedding.embed_query(pages_md_split[1].page_content)
vector2 = embedding.embed_query(pages_md_split[2].page_content)

In [ ]:
print(len(vector0))
print(len(vector1))
print(len(vector2))

In [ ]:
similarity_0_1 = np.dot(vector0, vector1)
similarity_1_2 = np.dot(vector1, vector2)
similarity_0_2 = np.dot(vector0, vector2)

In [ ]:
print('Similarity of the 0th and 1st markdown chunks:', similarity_0_1)
print('Similarity of the 1st and 2nd markdown chunks:', similarity_1_2)
print('Similarity of the 0th and 2nd markdown chunks:', similarity_0_2)
